# 6 · Théorie de l'information — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch06_information/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch06_information/03_notebook.ipynb`, créée par `python tools/start_chapter.py 6`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 6.1 à 6.6 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 6.12 | self_information et entropy | 🔨 | ★ | 15 |
|  | 6.13 | Distributions de caractères et de mots | 🔨 | ★★ | 20 |
|  | 6.14 | Qui a l'entropie par lettre la plus haute : Holmes ou Verne ? | 🔮 | ★★ | 15 |
|  | 6.15 | Fréquences des lettres en anglais et en français | 🎨 | ★★ | 20 |
| B | 6.16 | cross_entropy, kl_divergence et js_divergence | 🔨 | ★★ | 25 |
|  | 6.17 | La cross-entropy infinie : la lettre qui manque | 🐛 | ★★ | 20 |
|  | 6.18 | Coder le français avec le code de l'anglais, et l'inverse | 🔬 | ★★ | 25 |
|  | 6.19 | scipy.stats.entropy et un vrai compresseur (zlib) | 📦 | ★★ | 20 |
| C | 6.20 | Lire une courbe de loss : nats, bits et perplexité | 📈 | ★★ | 20 |
|  | 6.21 | Mesurer avant d'optimiser : compter des caractères vite | 🛠️ | ★★ | 20 |
|  | 6.22 | perplexity et log_loss | 🔨 | ★★ | 30 |
| D | 6.23 | Huffman : construire, encoder, décoder | 🔨 | ★★★ | 45 |
|  | 6.24 | Compresser Holmes : code fixe, Morse, Huffman et entropie | 🔬 | ★★★ | 30 |
|  | 6.25 | Le code de Huffman de Holmes pour envoyer Verne | 🔮 | ★★★ | 30 |
|  | 6.26 | Le contexte local réduit la surprise : les bigrammes | 🔬 | ★★★ | 40 |
|  | 6.27 | Passer sous la barre de Huffman lettre à lettre | 🏆 | ★★★ | 60 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="6")  # your own library

## Objectifs et rappel express

- Calculer la surprise d'un événement et l'entropie d'une distribution, en bits et en nats.
- Mesurer le coût d'un code fait pour une autre distribution : cross-entropy et divergence KL.
- Construire un code de Huffman, et comparer code fixe, Morse, Huffman, entropie et vrai compresseur.
- Relier cross-entropy, log loss et perplexité, et mesurer ce que le contexte fait gagner.

**Rappel express.** Surprise $I(p) = -\log_2 p$ ; entropie $H(p) = -\sum_i p_i \log_2 p_i$, entre 0 et $\log_2 n$ ; cross-entropy $H(p, q) = -\sum_i p_i \log_2 q_i \ge H(p)$ ; $\mathrm{KL}(p \,\|\, q) = H(p, q) - H(p) \ge 0$, non symétrique ; 1 nat $= \frac{1}{\ln 2} \approx 1{,}443$ bit ; perplexité $= e^{\text{loss en nats}} = 2^{\text{loss en bits}}$. En Python : `np.log2`, `np.log`, `collections.Counter`, `heapq`, `scipy.stats.entropy`, `zlib.compress`, `timeit.repeat`.

## Partie 0 · Vérifier tes exercices papier (✏️ 6.1 à 6.6)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[1, 2, 3]` ou `True`), pas l'expression Python, sinon tu ne vérifies rien. En Python, le séparateur décimal est un **point** (`0.125`) ; `0,125` sans guillemets serait un couple de deux nombres. Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les exercices ∂ 6.7 et 6.8 se corrigent avec `05_solutions.md`.

**Ex 6.1 — Combien de bits pour une pièce, un dé, une lettre E ?**

In [ ]:
# Ex 6.1: the values you found on paper
answer_6_1a = ...  # a) 'heads' with a fair coin, in bits
answer_6_1b = ...  # b) a 6 with a fair die, in bits (3 decimals)
answer_6_1c = ...  # c) an 'e' of probability 1/8, in bits
answer_6_1d = ...  # d) a 'z' of probability 1/1024, in bits
answer_6_1e = ...  # e) an 'e' then a 'z' (independent), in bits
answer_6_1f = ...  # f) the result (5, 2, 6) of three fair dice, in bits (3 decimals)
answer_6_1g = ...  # g) the 6 of b), in nats (3 decimals)
answer_6_1h = ...  # h) binary digits of a fixed-length code for a die
answer_6_1i = ...  # i) 'tails' with P(heads) = 0.9, in bits (3 decimals)

for letter, answer in zip("abcdefghi", [answer_6_1a, answer_6_1b, answer_6_1c, answer_6_1d, answer_6_1e, answer_6_1f, answer_6_1g, answer_6_1h, answer_6_1i]):
    wb.check(f"6.1{letter}", answer)

**Ex 6.2 — Bits par mot : Seuss, Holmes et l'alphabet**

In [ ]:
# Ex 6.2: the values you found on paper
answer_6_2a = ...  # a) bits per word for exactly 64 words
answer_6_2b = ...  # b) bits per word for 65 words
answer_6_2c = ...  # c) The Cat in the Hat, 236 words
answer_6_2d = ...  # d) Holmes, 7 819 words
answer_6_2e = ...  # e) 26 letters, the space and 10 digits
answer_6_2f = ...  # f) GPT-2, 50 257 tokens
answer_6_2g = ...  # g) bits to send the 105 849 words of Holmes with the code of d)
answer_6_2h = ...  # h) the largest vocabulary with 14 bits

for letter, answer in zip("abcdefgh", [answer_6_2a, answer_6_2b, answer_6_2c, answer_6_2d, answer_6_2e, answer_6_2f, answer_6_2g, answer_6_2h]):
    wb.check(f"6.2{letter}", answer)

**Ex 6.3 — Entropie de quelques distributions**

In [ ]:
# Ex 6.3: the values you found on paper
answer_6_3a = ...  # a) H of [1/2, 1/4, 1/8, 1/8], in bits
answer_6_3b = ...  # b) H of the uniform distribution on 8 outcomes
answer_6_3c = ...  # c) H of [0.9, 0.1] (3 decimals)
answer_6_3d = ...  # d) H of [1/2, 1/2, 0]
answer_6_3e = ...  # e) the entropy of a), in nats (3 decimals)
answer_6_3f = ...  # f) the largest entropy on 4 outcomes
answer_6_3g = ...  # g) the loaded die (6 half the time), 3 decimals

for letter, answer in zip("abcdefg", [answer_6_3a, answer_6_3b, answer_6_3c, answer_6_3d, answer_6_3e, answer_6_3f, answer_6_3g]):
    wb.check(f"6.3{letter}", answer)

**Ex 6.4 — Morse contre code fixe : SHERLOCK HOLMES**

In [ ]:
# Ex 6.4: the values you found on paper
answer_6_4a = ...  # a) symbols with the 5-symbol fixed code
answer_6_4b = ...  # b) dots and dashes in Morse
answer_6_4c = ...  # c) the ratio b) / a), 2 decimals
answer_6_4d = ...  # d) symbols with one silence between two consecutive letters
answer_6_4e = ...  # e) the ratio d) / a), 2 decimals
answer_6_4f = ...  # f) the number of ways to read four dots
answer_6_4g = ...  # g) is Morse without silences a prefix code? (True or False)

for letter, answer in zip("abcdefg", [answer_6_4a, answer_6_4b, answer_6_4c, answer_6_4d, answer_6_4e, answer_6_4f, answer_6_4g]):
    wb.check(f"6.4{letter}", answer)

**Ex 6.5 — Cross-entropy et KL dans les deux sens**

In [ ]:
# Ex 6.5: the values you found on paper
answer_6_5a = ...  # a) H(p), 3 decimals
answer_6_5b = ...  # b) H(p, q)
answer_6_5c = ...  # c) KL(p || q), 3 decimals
answer_6_5d = ...  # d) H(q, p), 3 decimals
answer_6_5e = ...  # e) KL(q || p), 3 decimals
answer_6_5f = ...  # f) is KL(p || q) < KL(q || p)? (True or False)
answer_6_5g = ...  # g) is H(p, r) finite for r = [1, 0]? (True or False)

for letter, answer in zip("abcdefg", [answer_6_5a, answer_6_5b, answer_6_5c, answer_6_5d, answer_6_5e, answer_6_5f, answer_6_5g]):
    wb.check(f"6.5{letter}", answer)

**Ex 6.6 — Un code de Huffman à la main**

In [ ]:
# Ex 6.6: the values you found on paper
answer_6_6a = ...  # a) bits per state with a fixed-length code
answer_6_6b = ...  # b) Huffman lengths [sun, clouds, rain, wind, snow], a list
answer_6_6c = ...  # c) the mean length, in bits per state
answer_6_6d = ...  # d) the entropy, 3 decimals
answer_6_6e = ...  # e) the Kraft sum
answer_6_6f = ...  # f) bits for sun, sun, rain, snow, clouds
answer_6_6g = ...  # g) the ratio c) / a), 3 decimals

for letter, answer in zip("abcdefg", [answer_6_6a, answer_6_6b, answer_6_6c, answer_6_6d, answer_6_6e, answer_6_6f, answer_6_6g]):
    wb.check(f"6.6{letter}", answer)

## Partie A · Mesurer l'information : surprise, entropie, lettres de Holmes et de Verne

Fiche §6.2 à §6.7. Tu écris les premières fonctions de `mylearn.info` (la surprise, l'entropie, les distributions de lettres et de mots), tu prévois puis mesures l'entropie des lettres en anglais et en français, et tu dessines leurs fréquences. La cellule ci-dessous charge les outils de tout le notebook et les deux livres : exécute-la d'abord.

In [ ]:
# Tools for the notebook exercises (parts A to D)
import collections
import itertools
import re
import timeit
import zlib

import matplotlib.pyplot as plt
import numpy as np
import scipy.stats


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def run_info_tests(keyword, impl="learner"):
    """Run the tests of mylearn.info selected by `keyword` (on YOUR code by default)."""
    command = [sys.executable, "-m", "pytest", "tests/test_ch06_info.py", "-k", keyword, "-q",
               "-p", "no:cacheprovider", "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix("FAILED tests/test_ch06_info.py::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


LETTERS = "abcdefghijklmnopqrstuvwxyz"            # the 26 letters, without accents
LETTERS_FR = LETTERS + "àâæçéèêëîïôœùûüÿ"         # plus the accented letters and ligatures of French (42 in all)
INDEX = {letter: i for i, letter in enumerate(LETTERS)}   # 'a' -> 0, ..., 'z' -> 25


def words(text):
    """The words of a text, in lower case: runs of letters (accented ones included); the rest separates them."""
    return re.findall(r"[^\W\d_]+", text.lower())


def letters_only(text):
    """The letters a to z of a text, in lower case and in order (accented letters, spaces, punctuation dropped)."""
    return "".join(ch for ch in text.lower() if "a" <= ch <= "z")


holmes = wb.datasets.load_holmes()      # ch. 1: The Adventures of Sherlock Holmes (English), a str
verne = wb.datasets.load_verne()        # ch. 1: Le Tour du monde en quatre-vingts jours (French), a str
print(f"holmes: {len(holmes)} characters, {len(words(holmes))} words · verne: {len(verne)} characters, "
      f"{len(words(verne))} words")

### Ex 6.12 — self_information et entropy 🔨 ★ ⏱️ 15 min
**Objectif :** écrire la surprise d'un événement et l'entropie d'une distribution, en bits ou en nats.  
**Prérequis :** Ex 6.1, Ex 6.3 · fiche §6.4, §6.7 · **Fil rouge :** synthétique · **mylearn :** `info.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/info.py` (créé par `python tools/start_chapter.py 6`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy et la bibliothèque standard sont permis (`np.asarray`, `np.log`, `np.log2`, `collections.Counter`, `heapq`…) ; SciPy, scikit-learn et PyTorch non : `scipy.stats.entropy`, `sklearn.metrics.log_loss` et `torch.nn.functional.cross_entropy` sont les **oracles** des tests. La cellule de vérification recharge ta librairie, affiche quelques valeurs, puis lance les tests de tes fonctions ; `python -m pytest tests/test_ch06_info.py -q` les lance tous dans un terminal.

Écris `self_information(p, base=2.0)` et `entropy(p, base=2.0)` (lis leurs docstrings) :
- deux petites fonctions d'aide te serviront dans tout le module : `_check_base(base)` lève une `ValueError` si la base n'est pas strictement positive ou vaut 1 ; `_as_distribution(p)` convertit `p` en tableau de flottants (`np.asarray(p, dtype=float)`) et lève une `ValueError` s'il n'est pas à une dimension, s'il est vide, s'il contient une valeur négative ou `NaN`, ou si sa somme s'écarte de 1 de plus de $10^{-6}$ (ne compare jamais une somme de flottants à 1 avec `==`) ;
- dans une base $b$ quelconque, $\log_b x = \frac{\ln x}{\ln b}$ (0B) ;
- `self_information` refuse une probabilité nulle, négative ou `NaN` (`~(p > 0)` attrape ces trois cas d'un coup), et une probabilité plus grande que 1 ; pour un `p` scalaire, elle renvoie un `float` Python, sinon un tableau de même forme ;
- `entropy` ignore les issues de probabilité nulle ($0 \log 0 = 0$) : garde seulement `p[p > 0]`, et renvoie un `float`.

La cellule de vérification affiche quelques valeurs, puis lance les tests des deux fonctions.

Dans tes notes : retrouve avec tes fonctions deux réponses de ✏️ 6.1 et deux de ✏️ 6.3. Que vaut l'entropie de $[0{,}9 ; 0{,}1]$ en nats (`base=np.e`) ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="6")   # reload your saved file
with wb.attempt("6.12"):
    surprise_12 = mylearn.info.self_information([1.0, 0.5, 0.125, 0.01])
    print("surprise of p = 1, 0.5, 0.125, 0.01 (bits):", np.round(surprise_12, 4))
    verdict("6.12", np.allclose(surprise_12, -np.log2([1.0, 0.5, 0.125, 0.01])), "les surprises en bits sont justes.",
            "self_information doit renvoyer -log2(p) pour chaque probabilité.")
    uniform_12, certain_12 = mylearn.info.entropy([0.25, 0.25, 0.25, 0.25]), mylearn.info.entropy([1.0, 0.0])
    verdict("6.12", abs(uniform_12 - 2) < 1e-12 and abs(certain_12) < 1e-12,
            "loi uniforme sur 4 issues : 2 bits ; issue certaine : 0 bit.",
            f"entropy([0.25] * 4) doit valoir 2 (j'obtiens {uniform_12!r}) et entropy([1, 0]) doit valoir 0 "
            f"(j'obtiens {certain_12!r}) : 0 log 0 = 0.")
    run_info_tests("test_self_information_ or test_entropy_")

### Ex 6.13 — Distributions de caractères et de mots 🔨 ★★ ⏱️ 20 min
**Objectif :** estimer la distribution des tokens d'un texte, sur un vocabulaire choisi, avec ou sans lissage.  
**Prérequis :** Ex 1.11 (fréquences des lettres) · Ex 6.12 · fiche §6.5, §6.6, §6.8 (lissage) · **Fil rouge :** Holmes/Verne · **mylearn :** `info.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/info.py`, mêmes règles qu'en 6.12 (NumPy et la bibliothèque standard permis, SciPy, scikit-learn et PyTorch non). Enregistre, puis relance la cellule de vérification.

Écris `token_distribution(tokens, vocabulary=None, smoothing=0.0)` et `char_distribution(text, alphabet=None, lowercase=True, smoothing=0.0)` (lis leurs docstrings) :
- compte les tokens avec `collections.Counter` (0A), en **un seul passage** : `tokens` peut être un générateur, qu'on ne peut lire qu'une fois ;
- sans `vocabulary`, le vocabulaire est la liste **triée** des tokens distincts ; avec, garde son ordre, ignore les autres tokens (ils ne comptent pas dans le total), et refuse un vocabulaire qui répète un token ;
- le **lissage de Laplace** (fiche §6.8) ajoute `smoothing` au compte de **chaque** token du vocabulaire, même s'il n'apparaît jamais ;
- `char_distribution` passe le texte en minuscules si `lowercase` (jamais l'alphabet), puis réutilise `token_distribution` : une chaîne est déjà une suite de caractères.

Puis, la cellule de vérification appelle tes fonctions, avec `words(text)` (fourni : la liste des mots d'un texte, en minuscules) :
a) la probabilité de « e » parmi les voyelles a, e, i, o, u de Holmes (`alphabet="aeiou"`) ;
b) le nombre de mots différents de Verne ;
c) la probabilité du mot « de » parmi tous les mots de Verne ;
d) la distribution des mots de Verne sur le vocabulaire `["the", "de", "and", "et"]`, dans cet ordre, avec `smoothing=1` (4 probabilités) ;
puis les tests des deux fonctions.

Dans tes notes : en d), pourquoi « the » a-t-il une probabilité non nulle ? Pourquoi « de » et « et » se partagent-ils presque toute la masse ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="6")   # reload your saved file
with wb.attempt("6.13"):
    _, vowels_13 = mylearn.info.char_distribution(holmes, alphabet="aeiou")
    wb.check("6.13a", vowels_13[1], computed=True)
    vocab_13, probs_13 = mylearn.info.token_distribution(words(verne))
    wb.check("6.13b", len(vocab_13), computed=True)
    wb.check("6.13c", probs_13[list(vocab_13).index("de")], computed=True)
    _, four_13 = mylearn.info.token_distribution(words(verne), vocabulary=["the", "de", "and", "et"], smoothing=1)
    wb.check("6.13d", four_13, computed=True)
    run_info_tests("test_token_distribution_ or test_char_distribution_")

### Ex 6.14 — Qui a l'entropie par lettre la plus haute : Holmes ou Verne ? 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir puis mesurer l'entropie des lettres de deux langues, et voir qu'elle dépend de l'alphabet choisi.  
**Prérequis :** Ex 6.13 · fiche §6.7 · **Fil rouge :** Holmes/Verne · **Parcours :** C

On compare l'entropie de la distribution des lettres de Holmes (anglais) et de Verne (français), de deux façons :
- sur les 26 lettres `LETTERS` (a à z) : comme au ch. 1 (1.11), les lettres accentuées ne comptent pas ;
- sur les 42 lettres `LETTERS_FR` : les 26 lettres, plus les lettres accentuées et les ligatures du français (à, é, ç, œ…), chacune comptée comme une lettre à part.

**Avant d'exécuter quoi que ce soit**, prédis :
a) `prediction_6_14a` : sur les 26 lettres, l'entropie de Holmes est-elle plus haute que celle de Verne ? (`True` ou `False`)
b) `prediction_6_14b` : sur les 42 lettres, l'entropie de Holmes est-elle plus haute que celle de Verne ? (`True` ou `False`)
c) `prediction_6_14c` : l'entropie de Holmes sur les 26 lettres est la plus proche de quelle valeur : 3, 4, 4,7 ou 5 bits ?

Écris ton hypothèse (cellule 📝), puis tes trois prédictions. Ensuite seulement, exécute l'**Expérience** : elle calcule avec tes fonctions d) l'entropie de Holmes sur 26 lettres, e) celle de Verne sur 26 lettres, f) celle de Verne sur 42 lettres.

Dans tes notes, après l'expérience : explique ce que tu observes sur 26 lettres, puis sur 42. Que vaut l'entropie la plus grande possible sur 26 lettres ?

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_6_14a = ...   # a) True or False
prediction_6_14b = ...   # b) True or False
prediction_6_14c = ...   # c) 3, 4, 4.7 or 5

In [ ]:
wb.check("6.14a", prediction_6_14a)
wb.check("6.14b", prediction_6_14b)
wb.check("6.14c", prediction_6_14c)

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="6")   # reload your saved file
if any(answer is ... or answer is None for answer in [prediction_6_14a, prediction_6_14b, prediction_6_14c]):
    print("✍️ Écris d'abord tes trois prédictions dans la cellule du dessus.")
else:
    with wb.attempt("6.14"):
        entropies_14 = {}
        for name_14, text_14, alphabet_14 in [("Holmes, 26 letters", holmes, LETTERS), ("Verne, 26 letters", verne, LETTERS),
                                              ("Holmes, 42 letters", holmes, LETTERS_FR),
                                              ("Verne, 42 letters", verne, LETTERS_FR)]:
            _, probs_14 = mylearn.info.char_distribution(text_14, alphabet=alphabet_14)
            entropies_14[name_14] = mylearn.info.entropy(probs_14)
            print(f"{name_14:20s} {entropies_14[name_14]:.4f} bits per letter")
        print(f"the largest possible: {np.log2(26):.4f} bits on 26 letters, {np.log2(42):.4f} on 42")
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.barh(list(entropies_14), list(entropies_14.values()), color=["C0", "C1", "C0", "C1"])
        ax.axvline(np.log2(26), ls="--", color="gray", lw=1)
        ax.set(xlabel="entropy (bits per letter)", xlim=(3.5, 5.5))
        ax.invert_yaxis()
        plt.show()
        print(f"your predictions: a) {prediction_6_14a}   b) {prediction_6_14b}   c) {prediction_6_14c}")
        wb.check("6.14d", entropies_14["Holmes, 26 letters"], computed=True)
        wb.check("6.14e", entropies_14["Verne, 26 letters"], computed=True)
        wb.check("6.14f", entropies_14["Verne, 42 letters"], computed=True)

### Ex 6.15 — Fréquences des lettres en anglais et en français 🎨 ★★ ⏱️ 20 min
**Objectif :** tracer côte à côte les distributions des lettres de deux textes, triées, pour les comparer d'un coup d'œil.  
**Prérequis :** Ex 6.13 · fiche §6.6 · livre §6.6 (figures 6.6 et 6.7) · **Fil rouge :** Holmes/Verne · **Parcours :** C

Le livre trace les fréquences des lettres de *Treasure Island* (figures 6.6 et 6.7). Fais-le pour Holmes **et** Verne, côte à côte. Les deux distributions sont fournies, sur les 26 lettres (`p_holmes_15`, `p_verne_15`).

Écris `draw_letters_15(ax, alphabet, p_holmes, p_verne)` qui trace, sur l'axe `ax`, des **barres horizontales groupées** :
- une ligne par lettre, deux barres par ligne (Holmes et Verne, deux couleurs) ;
- les lettres **triées par fréquence décroissante dans Holmes**, la plus fréquente en haut : `order = np.argsort(-p_holmes)`, puis `ax.invert_yaxis()` ;
- les barres de la ligne $i$ décalées de $\pm 0{,}2$ : `ax.barh(y - 0.2, …, height=0.4, label="Holmes")`, puis la même chose pour Verne à `y + 0.2` ;
- les lettres en étiquettes de l'axe des $y$ (`ax.set_yticks(y)`, `ax.set_yticklabels(...)`), la probabilité en $x$, un titre et une légende.

Dans tes notes : quelles lettres changent le plus de rang d'une langue à l'autre ? Le Morse, conçu pour l'anglais, est-il adapté au français ?

In [ ]:
counts_holmes_15 = collections.Counter(letters_only(holmes))      # plain counts: no mylearn needed here
counts_verne_15 = collections.Counter(letters_only(verne))
p_holmes_15 = np.array([counts_holmes_15[c] for c in LETTERS], dtype=float)
p_holmes_15 /= p_holmes_15.sum()
p_verne_15 = np.array([counts_verne_15[c] for c in LETTERS], dtype=float)
p_verne_15 /= p_verne_15.sum()

In [ ]:
def draw_letters_15(ax, alphabet, p_holmes, p_verne):
    """Grouped horizontal bars of two letter distributions, sorted by p_holmes, the most frequent on top."""
    raise NotImplementedError("draw_letters_15() is not written yet")

In [ ]:
with wb.attempt("6.15"):
    fig, ax = plt.subplots(figsize=(7, 8))
    try:
        draw_letters_15(ax, LETTERS, p_holmes_15, p_verne_15)
    except BaseException:
        plt.close(fig)                 # no empty figure while draw_letters_15 is not written
        raise
    widths_15 = sorted(round(bar.get_width(), 9) for bar in ax.patches)
    expected_15 = sorted(round(p, 9) for p in np.concatenate([p_holmes_15, p_verne_15]))
    verdict("6.15", widths_15 == expected_15, "52 barres, deux par lettre, aux bonnes longueurs.",
            f"il faut 52 barres horizontales (2 × 26) dont les longueurs sont les probabilités ; j'en compte "
            f"{len(ax.patches)}.")
    positions_15 = ax.get_yticks()
    labels_15 = [label.get_text() for label in ax.get_yticklabels()]
    from_top_15 = [label for _, label in sorted(zip(positions_15, labels_15), reverse=not ax.yaxis_inverted())]
    expected_order_15 = [LETTERS[i] for i in np.argsort(-p_holmes_15, kind="stable")]
    verdict("6.15", from_top_15 == expected_order_15,
            "les lettres sont triées par fréquence dans Holmes, la plus fréquente en haut.",
            f"de haut en bas, j'attends les lettres triées par fréquence dans Holmes ({' '.join(expected_order_15[:5])}…), "
            f"je lis {' '.join(from_top_15[:5])}…")
    verdict("6.15", ax.get_legend() is not None, "la légende est là.", "ajoute une légende (ax.legend()).")
    plt.show()

## Partie B · Coder avec le mauvais code : cross-entropy, KL et compression

Fiche §6.8 et §6.9, puis « au-delà du livre » (2). Tu écris la cross-entropy et les divergences, tu répares une cross-entropy infinie, tu mesures ce que coûte d'envoyer le français avec le code de l'anglais, et tu compares l'entropie à un vrai compresseur.

### Ex 6.16 — cross_entropy, kl_divergence et js_divergence 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire la cross-entropy, la divergence KL et celle de Jensen-Shannon, avec leurs cas infinis.  
**Prérequis :** Ex 6.12 · Ex 6.5 · fiche §6.8, §6.9 · **Fil rouge :** synthétique · **mylearn :** `info.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/info.py`, mêmes règles qu'en 6.12 (NumPy et la bibliothèque standard permis, SciPy, scikit-learn et PyTorch non). Enregistre, puis relance la cellule de vérification.

Écris `cross_entropy(p, q, base=2.0)`, `kl_divergence(p, q, base=2.0)` et `js_divergence(p, q, base=2.0)` (lis leurs docstrings) :
- valide les deux distributions avec ton `_as_distribution` (6.12), et vérifie qu'elles ont la même forme ;
- ne somme que sur les issues où $p_i > 0$ ; si $q_i = 0$ pour l'une d'elles, renvoie `float("inf")`. Teste ce cas **avant** de prendre le logarithme : ni `NaN`, ni avertissement de NumPy ;
- `kl_divergence` renvoie $\sum p_i \log \frac{p_i}{q_i}$ ; les arrondis peuvent donner $-10^{-17}$ quand $q$ est presque $p$ : renvoie alors 0 ;
- `js_divergence` construit le mélange $m = \frac{p + q}{2}$, puis réutilise deux fois `kl_divergence`.

La vérification affiche les valeurs de ✏️ 6.5 calculées par tes fonctions, contrôle deux propriétés, puis lance les tests.

Dans tes notes : pourquoi la divergence de Jensen-Shannon reste-t-elle finie quand la KL est infinie ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="6")   # reload your saved file
with wb.attempt("6.16"):
    p_16, q_16 = [0.8, 0.2], [0.5, 0.5]
    ce_16 = mylearn.info.cross_entropy(p_16, q_16)
    kl_16, kl_back_16 = mylearn.info.kl_divergence(p_16, q_16), mylearn.info.kl_divergence(q_16, p_16)
    print(f"H(p, q) = {ce_16:.4f}   KL(p || q) = {kl_16:.4f}   KL(q || p) = {kl_back_16:.4f}   "
          f"JS(p, q) = {mylearn.info.js_divergence(p_16, q_16):.4f}")
    verdict("6.16", abs(ce_16 - (mylearn.info.entropy(p_16) + kl_16)) < 1e-12, "H(p, q) = H(p) + KL(p || q) (∂ 6.7).",
            "H(p, q) doit être égale à H(p) + KL(p || q) (∂ 6.7) : vérifie les deux fonctions.")
    infinite_16 = mylearn.info.cross_entropy([0.5, 0.5], [1.0, 0.0])
    verdict("6.16", infinite_16 == float("inf"), "une issue jugée impossible par q donne une cross-entropy infinie.",
            f"cross_entropy([0.5, 0.5], [1, 0]) doit renvoyer inf, j'obtiens {infinite_16!r}.")
    run_info_tests("test_cross_entropy_ or test_kl_divergence_ or test_js_divergence_")

### Ex 6.17 — La cross-entropy infinie : la lettre qui manque 🐛 ★★ ⏱️ 20 min
**Objectif :** diagnostiquer une cross-entropy infinie, et la corriger par un lissage placé au bon endroit.  
**Prérequis :** Ex 6.16 · fiche §6.8 (lissage de Laplace) · livre §6.8.2 · **Fil rouge :** Holmes/Verne · **Parcours :** C

Une collègue veut savoir combien de bits par lettre coûte l'envoi de Verne avec un code construit sur les lettres de Holmes. Pour garder les accents du français, elle prend l'alphabet `LETTERS_FR` (42 lettres). Son code, ci-dessous, affiche… `inf`. Trouve pourquoi, puis corrige.
a) `n_missing_17` : combien de lettres de `LETTERS_FR` apparaissent dans Verne mais jamais dans Holmes ?
b) `worst_count_17` : le nombre d'apparitions, dans Verne, de la plus fréquente de ces lettres ;
c) `share_17` : la proportion des lettres de Verne (parmi les 42 de `LETTERS_FR`) qui sont des lettres de a) (4 décimales) ;
d) `ce_fixed_17` : corrige le calcul avec le lissage de Laplace (`smoothing=1`), **à un seul endroit**, et donne le nombre de bits par lettre obtenu (4 décimales).

Le cœur de l'exercice est le choix de l'endroit : lisser la distribution de Verne, celle de Holmes, ou les deux ? Que se passe-t-il dans chaque cas ?

In [ ]:
# The code of a colleague: the bits per letter to send Verne with a code built on the letters of Holmes
with wb.attempt("6.17"):
    _, p_verne_17 = mylearn.info.char_distribution(verne, alphabet=LETTERS_FR)
    _, q_holmes_17 = mylearn.info.char_distribution(holmes, alphabet=LETTERS_FR)
    print("bits per letter to send Verne with the code of Holmes:", mylearn.info.cross_entropy(p_verne_17, q_holmes_17))

In [ ]:
n_missing_17 = ...     # a) an integer
worst_count_17 = ...   # b) an integer
share_17 = ...         # c)
ce_fixed_17 = ...      # d) the corrected cross-entropy, in bits per letter

In [ ]:
wb.check("6.17a", n_missing_17)
wb.check("6.17b", worst_count_17)
wb.check("6.17c", share_17)
wb.check("6.17d", ce_fixed_17)

### Ex 6.18 — Coder le français avec le code de l'anglais, et l'inverse 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer le surcoût d'un code fait pour une autre langue, dans les deux sens, et voir l'asymétrie de la KL.  
**Prérequis :** Ex 6.13 · Ex 6.16 · fiche §6.8, §6.9 · livre §6.8.2, §6.9 · **Fil rouge :** Holmes/Verne · **Parcours :** R, C

Le livre envoie *Treasure Island* avec le code de *Huckleberry Finn*. Fais de même avec deux langues. Sur les 26 lettres (`LETTERS`, accents ignorés, toutes présentes dans les deux livres), avec tes fonctions (4 décimales) :
a) `ce_verne_holmes_18` : le nombre de bits par lettre pour envoyer Verne avec le code idéal de Holmes, $H(p_{\text{Verne}}, q_{\text{Holmes}})$ ;
b) `ce_holmes_verne_18` : l'inverse, Holmes avec le code de Verne ;
c) `kl_verne_holmes_18` : $\mathrm{KL}(\text{Verne} \,\|\, \text{Holmes})$, le surcoût de a) par rapport au code idéal de Verne ;
d) `kl_holmes_verne_18` : $\mathrm{KL}(\text{Holmes} \,\|\, \text{Verne})$ ;
e) `js_18` : la divergence de Jensen-Shannon entre les deux distributions ;
f) `kl_halves_18` : la KL entre la première et la seconde moitié du **texte** de Holmes, `holmes[:len(holmes) // 2]` et `holmes[len(holmes) // 2:]`, dans cet ordre (5 décimales).

Une fois c) et d) remplis, la cellule suivante trace la contribution de chaque lettre à ces deux KL.

Dans tes notes : quel envoi coûte le plus cher, et pourquoi ? Quelles lettres pèsent le plus dans d) ? Que dit f) ?

In [ ]:
ce_verne_holmes_18 = ...   # a) Verne sent with the code of Holmes
ce_holmes_verne_18 = ...   # b) Holmes sent with the code of Verne
kl_verne_holmes_18 = ...   # c)
kl_holmes_verne_18 = ...   # d)
js_18 = ...                # e)
kl_halves_18 = ...         # f) first half of the text of Holmes || second half

In [ ]:
wb.check("6.18a", ce_verne_holmes_18)
wb.check("6.18b", ce_holmes_verne_18)
wb.check("6.18c", kl_verne_holmes_18)
wb.check("6.18d", kl_holmes_verne_18)
wb.check("6.18e", js_18)
wb.check("6.18f", kl_halves_18)

In [ ]:
if filled(kl_verne_holmes_18, kl_holmes_verne_18):
    with wb.attempt("6.18"):
        _, p_h_18 = mylearn.info.char_distribution(holmes, alphabet=LETTERS)
        _, p_v_18 = mylearn.info.char_distribution(verne, alphabet=LETTERS)
        fig, ax = plt.subplots(figsize=(11, 3.6))
        x_18 = np.arange(26)
        ax.bar(x_18 - 0.2, p_v_18 * np.log2(p_v_18 / p_h_18), width=0.4, label="KL(Verne || Holmes): Verne sent with the code of Holmes")
        ax.bar(x_18 + 0.2, p_h_18 * np.log2(p_h_18 / p_v_18), width=0.4, label="KL(Holmes || Verne): Holmes sent with the code of Verne")
        ax.axhline(0, color="black", lw=0.8)
        ax.set_xticks(x_18)
        ax.set_xticklabels(list(LETTERS))
        ax.set(ylabel="contribution (bits per letter)", title="Each letter's term p_i log2(p_i / q_i): the sum is the KL")
        ax.legend(fontsize=8)
        plt.show()
else:
    print("⏳ Ex 6.18 : remplis d'abord c) et d) ; la figure des contributions s'affichera ici.")

### Ex 6.19 — scipy.stats.entropy et un vrai compresseur (zlib) 📦 ★★ ⏱️ 20 min
**Objectif :** utiliser `scipy.stats.entropy` et un compresseur de la bibliothèque standard, et voir qu'un compresseur exploite le contexte.  
**Prérequis :** Ex 6.13 · fiche §6.7, au-delà du livre (2) (🧮 compression) · livre §6.2.2 · **Fil rouge :** Holmes · **Parcours :** C

1. `scipy.stats.entropy(pk, qk=None, base=None)` calcule l'entropie, et la KL si on lui donne `qk`. Lis sa documentation (`help(scipy.stats.entropy)`) : accepte-t-elle des **comptes** au lieu de probabilités ? Quelle base utilise-t-elle par défaut ?
a) `h_chars_19` : l'entropie, en bits par caractère, de la distribution de **tous** les caractères de Holmes, tels quels (majuscules, espaces, ponctuation, retours à la ligne, sans passer en minuscules), avec `scipy.stats.entropy` appliquée aux comptes de `collections.Counter(holmes)` (4 décimales).

2. Le module `zlib` de la bibliothèque standard comprime sans perte (fiche, 🧮 compression) : `zlib.compress(data, 9)` prend des octets (`holmes.encode("utf-8")`) et renvoie des octets, au niveau de compression 9 (le plus fort).
b) `zlib_bits_19` : le nombre de **bits** du texte comprimé, divisé par le nombre de **caractères** de Holmes (1 décimale) ;
c) `shuffled_bits_19` : la même mesure sur Holmes **mélangé** : exactement les mêmes caractères, dans un ordre aléatoire, `"".join(np.random.default_rng(19).permutation(list(holmes)))` (1 décimale) ;
d) `zlib_wins_19` : sur le vrai texte, zlib fait-il mieux que l'entropie a) ? (`True` ou `False`)

Dans tes notes : comment zlib peut-il descendre sous l'entropie a), qui suppose chaque caractère tiré indépendamment des autres ? Pourquoi fait-il moins bien sur le texte mélangé, qui a pourtant exactement la même distribution de caractères ?

In [ ]:
h_chars_19 = ...         # a)
zlib_bits_19 = ...       # b)
shuffled_bits_19 = ...   # c)
zlib_wins_19 = ...       # d) True or False

In [ ]:
wb.check("6.19a", h_chars_19)
wb.check("6.19b", zlib_bits_19)
wb.check("6.19c", shuffled_bits_19)
wb.check("6.19d", zlib_wins_19)

## Partie C · Bits, nats et perplexité : la loss des modèles de langage

Fiche, au-delà du livre (1). Tu lis une courbe de loss en nats, tu mesures la vitesse de plusieurs façons de compter des lettres, puis tu écris la perplexité et la log loss, la loss de presque tous les classifieurs.

### Ex 6.20 — Lire une courbe de loss : nats, bits et perplexité 📈 ★★ ⏱️ 20 min
**Objectif :** lire une courbe de loss en nats, la convertir en bits et en perplexité, et repérer le surapprentissage.  
**Prérequis :** Ex 6.16 · fiche, au-delà du livre (1) · **Fil rouge :** synthétique · **Parcours :** R, M

Un modèle de langage au niveau des caractères (il prédit le caractère suivant) a été entraîné pendant 3 000 pas. On a noté sa loss, la cross-entropy en **nats** comme dans PyTorch, sur les données d'entraînement (`train_20`) et sur des données de validation (`val_20`), tous les 25 pas (`steps_20`). Lis la figure, puis réponds avec les tableaux :
a) `vocab_20` : au pas 0, le modèle n'a rien appris et répartit ses probabilités uniformément entre tous les caractères possibles. Combien de caractères y a-t-il ? (un entier)
b) `best_step_20` : le pas où la loss de **validation** est la plus basse ;
c) `best_bits_20` : cette loss minimale, en bits par caractère (3 décimales) ;
d) `best_ppl_20` : la perplexité correspondante (2 décimales) ;
e) `ppl3_step_20` : le premier pas où la perplexité de validation passe sous 3 ;
f) `last_ppl_20` : la perplexité de validation au dernier pas (2 décimales).

Dans tes notes : que se passe-t-il après le pas b) ? Quel nom porte ce phénomène (ch. 9), et que ferais-tu ?

In [ ]:
steps_20 = np.arange(0, 3001, 25)                     # the training steps where the loss was measured
decay_20 = 1.05 + (np.log(64) - 1.05) * np.exp(-steps_20 / 320) - 0.000035 * steps_20
train_20 = decay_20 + 0.003 * np.sin(steps_20 / 41)                                # loss on the training data (nats)
val_20 = (decay_20 + 0.03 * (1 - np.exp(-steps_20 / 300))                          # loss on validation data (nats)
          + 0.25 * np.clip((steps_20 - 1400) / 1600, 0, None) ** 2 + 0.004 * np.sin(steps_20 / 53))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(steps_20, train_20, label="training loss")
ax.plot(steps_20, val_20, label="validation loss")
ax.set(xlabel="training step", ylabel="cross-entropy (nats per character)", ylim=(0.8, 4.3))
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
vocab_20 = ...       # a) an integer
best_step_20 = ...   # b)
best_bits_20 = ...   # c)
best_ppl_20 = ...    # d)
ppl3_step_20 = ...   # e)
last_ppl_20 = ...    # f)

In [ ]:
wb.check("6.20a", vocab_20)
wb.check("6.20b", best_step_20)
wb.check("6.20c", best_bits_20)
wb.check("6.20d", best_ppl_20)
wb.check("6.20e", ppl3_step_20)
wb.check("6.20f", last_ppl_20)

### Ex 6.21 — Mesurer avant d'optimiser : compter des caractères vite 🛠️ ★★ ⏱️ 20 min
**Objectif :** mesurer la durée de plusieurs implémentations avec `timeit`, avant de choisir laquelle garder.  
**Prérequis :** Ex 6.13 · 0A (`Counter`, NumPy) · fiche §6.5 · **Fil rouge :** Holmes · **Parcours :** C

Avant d'optimiser un calcul, un professionnel **mesure**. Écris quatre fonctions qui renvoient le tableau des comptes des 26 lettres (dans l'ordre de `LETTERS`, des entiers) d'un texte déjà en minuscules :
1. `count_loop_21(text)` : une boucle `for` sur les caractères et un dictionnaire ;
2. `count_counter_21(text)` : `collections.Counter(text)`, puis les 26 comptes ;
3. `count_str_21(text)` : la méthode `text.count(letter)`, une fois par lettre (26 passages sur le texte, mais écrits en C) ;
4. `count_numpy_21(text)` : les octets du texte, `np.frombuffer(text.encode("ascii", "ignore"), dtype=np.uint8)`, puis `np.bincount(…, minlength=123)` et la tranche des codes 97 à 122 (`ord("a")` à `ord("z")`).

Avant de lancer la mesure, note dans `06_mes_reponses.md` laquelle tu crois la plus rapide. La cellule de vérification compare chaque résultat aux comptes de `Counter`, puis mesure chaque fonction avec `timeit.repeat(lambda: f(text), number=3, repeat=5)` : elle garde le **minimum** des cinq mesures, divisé par 3.

Dans tes notes : laquelle est la plus rapide, et de combien ? Était-ce ta prédiction ? Pourquoi garder le minimum de plusieurs mesures plutôt qu'une seule ? Que mesure `time.perf_counter()`, et pourquoi `timeit` l'utilise-t-il ?

In [ ]:
holmes_lower_21 = holmes.lower()        # counted once lowered: we only time the counting

In [ ]:
def count_loop_21(text):
    """The 26 counts of the letters a-z in text (already lower-cased), with a for loop and a dict."""
    raise NotImplementedError("count_loop_21() is not written yet")


def count_counter_21(text):
    """The same 26 counts, with collections.Counter."""
    raise NotImplementedError("count_counter_21() is not written yet")


def count_str_21(text):
    """The same 26 counts, with text.count(letter)."""
    raise NotImplementedError("count_str_21() is not written yet")


def count_numpy_21(text):
    """The same 26 counts, with np.frombuffer and np.bincount."""
    raise NotImplementedError("count_numpy_21() is not written yet")

In [ ]:
def check_counts_21(name, function):
    """The 26 counts of `function` on holmes_lower_21, compared with collections.Counter; then the best time."""
    counts = np.asarray(function(holmes_lower_21))
    ok = counts.shape == (26,) and np.array_equal(counts, reference_21)
    verdict("6.21", ok, f"{name} : les 26 comptes sont justes.",
            f"{name} doit renvoyer les 26 comptes, dans l'ordre de LETTERS (forme {counts.shape}).")
    seconds = min(timeit.repeat(lambda: function(holmes_lower_21), number=3, repeat=5)) / 3
    return seconds if ok else None


counter_21 = collections.Counter(holmes_lower_21)
reference_21 = np.array([counter_21[c] for c in LETTERS])
times_21 = {}
for name_21, function_21 in [("count_loop_21", count_loop_21), ("count_counter_21", count_counter_21),
                             ("count_str_21", count_str_21), ("count_numpy_21", count_numpy_21)]:
    with wb.attempt(f"6.21 ({name_21})"):
        times_21[name_21] = check_counts_21(name_21, function_21)
done_21 = {name: seconds for name, seconds in times_21.items() if seconds is not None}
if done_21:
    fastest_21 = min(done_21.values())
    for name_21, seconds_21 in sorted(done_21.items(), key=lambda item: item[1]):
        print(f"{name_21:18s} {1000 * seconds_21:8.2f} ms   ({seconds_21 / fastest_21:5.1f} × the fastest)")

### Ex 6.22 — perplexity et log_loss 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire la perplexité d'un modèle et la log loss d'un classifieur, et les appliquer à un modèle de lettres.  
**Prérequis :** Ex 6.16 · ch. 3 (probabilités prédites, calibration) · fiche, au-delà du livre (1) · **Fil rouge :** Holmes · **mylearn :** `info.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/info.py`, mêmes règles qu'en 6.12 (NumPy et la bibliothèque standard permis, SciPy, scikit-learn et PyTorch non). Enregistre, puis relance la cellule de vérification.

Écris `perplexity(token_probs)` et `log_loss(y_true, y_prob, base=np.e)` (lis leurs docstrings) :
- `perplexity` : l'exponentielle de la moyenne des $-\ln p_i$, où $p_i$ est la probabilité que le modèle donnait au token réellement observé ; elle ne dépend pas de la base (fiche, au-delà du livre (1)) ;
- `log_loss` : pour chaque exemple, la probabilité que le modèle a donnée à la **vraie** classe (en binaire, `y_prob` est la probabilité de la classe 1, donc celle de la classe 0 vaut `1 - y_prob`), coupée (`np.clip`, sur une copie) dans $[\varepsilon ; 1 - \varepsilon]$ avec $\varepsilon$ = `np.finfo(float).eps`, puis la moyenne des $-\log$ dans la base demandée (des **nats** par défaut, comme scikit-learn) ;
- valide les entrées comme le disent les docstrings : longueurs, étiquettes entières dans le bon intervalle, probabilités dans $[0 ; 1]$ (ce qui exclut `NaN`), lignes de somme 1.

La vérification appelle tes fonctions avec le modèle de lettres de Holmes `model_22` (fourni : les probabilités des 26 lettres sur tout le livre, lissées) :
a) la perplexité de ce modèle sur les lettres de la phrase anglaise `english_22` ;
b) sur sa traduction française, `french_22` ;
c) sur le pangramme anglais `pangram_22` (une phrase qui contient les 26 lettres) ;
puis, avec `log_loss` :
d) la log loss, en nats, des prédictions binaires `p_22` pour les vraies classes `y_22` ;
e) la log loss, en **bits**, des prédictions à trois classes `P3_22` pour `y3_22` ;
et lance enfin les tests des deux fonctions.

Dans tes notes : pourquoi la phrase française surprend-elle davantage le modèle de Holmes ? Pourquoi le pangramme le surprend-il encore plus, alors qu'il est en anglais ?

In [ ]:
counts_22 = collections.Counter(letters_only(holmes))
model_22 = np.array([counts_22[c] + 1 for c in LETTERS], dtype=float)       # Laplace smoothing (6.13)
model_22 /= model_22.sum()                                                   # the probability of each letter, in Holmes
english_22 = "The train leaves London at a quarter to nine."
french_22 = "Le train quitte Londres à neuf heures moins le quart."
pangram_22 = "The quick brown fox jumps over the lazy dog."


def letter_probs_22(sentence):
    """The probability that model_22 gives to each letter a-z of the sentence, in order."""
    return model_22[[INDEX[c] for c in letters_only(sentence)]]


y_22, p_22 = [1, 0, 1, 1, 0, 1], [0.9, 0.2, 0.6, 0.95, 0.4, 0.3]               # binary: probability of class 1
y3_22 = [0, 2, 1, 2]
P3_22 = [[0.7, 0.2, 0.1], [0.1, 0.3, 0.6], [0.25, 0.5, 0.25], [0.2, 0.2, 0.6]]   # one row of 3 probabilities per sample

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="6")   # reload your saved file
with wb.attempt("6.22"):
    wb.check("6.22a", mylearn.info.perplexity(letter_probs_22(english_22)), computed=True)
    wb.check("6.22b", mylearn.info.perplexity(letter_probs_22(french_22)), computed=True)
    wb.check("6.22c", mylearn.info.perplexity(letter_probs_22(pangram_22)), computed=True)
    wb.check("6.22d", mylearn.info.log_loss(y_22, p_22), computed=True)
    wb.check("6.22e", mylearn.info.log_loss(y3_22, P3_22, base=2), computed=True)
    run_info_tests("test_perplexity_ or test_log_loss_")

## Partie D · Huffman, compression et contexte

Fiche §6.6, §6.7, puis « au-delà du livre » (2). Tu écris le code de Huffman, tu compresses Holmes de plusieurs façons, tu prévois le coût d'un code fait pour une autre langue, puis tu mesures ce que le contexte local fait gagner, jusqu'à battre le code de Huffman lettre à lettre.

### Ex 6.23 — Huffman : construire, encoder, décoder 🔨 ★★★ ⏱️ 45 min
**Objectif :** construire un code de Huffman avec une file de priorité, puis encoder et décoder un message.  
**Prérequis :** Ex 6.6 · Ex 6.13 · 0A (`heapq`, §100.6.6) · fiche §6.6 · **Fil rouge :** Holmes · **mylearn :** `info.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/info.py`, mêmes règles qu'en 6.12 (NumPy et la bibliothèque standard permis, SciPy, scikit-learn et PyTorch non). Enregistre, puis relance la cellule de vérification.

Écris `huffman_code(symbols, probs)`, `huffman_encode(symbols, code)` et `huffman_decode(bits, code)` (lis leurs docstrings) :
- `huffman_code` : valide les entrées (longueurs, symboles répétés, distribution), puis remplis une file de priorité `heapq` (0A, §100.6.6) de triplets `(probabilité, numéro, groupe de symboles)`. Tant qu'il reste plus d'un groupe, retire les **deux moins probables**, ajoute un `"0"` **devant** le mot de code de chaque symbole du premier groupe et un `"1"` devant ceux du second, puis remets leur fusion dans la file, avec un nouveau numéro. Le numéro, un compteur qui augmente, départage les égalités de probabilité : sans lui, Python comparerait deux groupes. Un symbole seul reçoit `"0"` ;
- `huffman_encode` : la concaténation des mots de code (`"".join`) ; un symbole sans mot de code lève une `ValueError` (pas une `KeyError`) ;
- `huffman_decode` : vérifie que les bits ne contiennent que des `"0"` et des `"1"` et que le code est préfixe, puis lis les bits un par un : ajoute chacun au mot en cours ; dès que celui-ci est un mot de code, écris son symbole et repars d'un mot vide. À la fin, il ne doit rester aucun bit en attente.

La vérification construit le code des lettres de Holmes avec tes fonctions, encode puis décode le début du livre, puis lance les tests.

Dans tes notes : quelles lettres ont les mots de code les plus courts, et les plus longs ? Compare avec le Morse (✏️ 6.4).

In [ ]:
counts_23 = collections.Counter(letters_only(holmes))
p_holmes_23 = np.array([counts_23[c] for c in LETTERS], dtype=float)
p_holmes_23 /= p_holmes_23.sum()                       # the letters of Holmes (plain counts, no mylearn needed)

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="6")   # reload your saved file
with wb.attempt("6.23"):
    code_23 = mylearn.info.huffman_code(list(LETTERS), p_holmes_23)
    words_23 = sorted(code_23.values())
    prefix_free_23 = not any(b.startswith(a) for a, b in zip(words_23, words_23[1:]))
    kraft_23 = sum(2.0 ** -len(word) for word in code_23.values())
    verdict("6.23", prefix_free_23 and abs(kraft_23 - 1) < 1e-12, "le code des lettres de Holmes est préfixe et complet (Kraft = 1).",
            f"le code doit être préfixe et sa somme de Kraft valoir 1 (j'obtiens {kraft_23!r}).")
    sample_23 = letters_only(holmes[:20000])
    bits_23 = mylearn.info.huffman_encode(sample_23, code_23)
    back_23 = "".join(mylearn.info.huffman_decode(bits_23, code_23))
    verdict("6.23", back_23 == sample_23, f"{len(sample_23)} lettres encodées en {len(bits_23)} bits, puis décodées à l'identique.",
            "le décodage ne redonne pas les lettres encodées.")
    by_length_23 = sorted(code_23, key=lambda letter: (len(code_23[letter]), letter))
    print("shortest codewords:", {c: code_23[c] for c in by_length_23[:4]}, "· longest:", {c: code_23[c] for c in by_length_23[-3:]})
    run_info_tests("test_huffman_")

### Ex 6.24 — Compresser Holmes : code fixe, Morse, Huffman et entropie 🔬 ★★★ ⏱️ 30 min
**Objectif :** comparer le coût d'un même texte avec un code fixe, le Morse, le code de Huffman et la borne de l'entropie.  
**Prérequis :** Ex 6.23 · Ex 6.4 · fiche §6.6, §6.7 · livre §6.6 · **Fil rouge :** Holmes · **Parcours :** M, C

Toutes les lettres a–z de Holmes, dans l'ordre, sans espaces ni ponctuation (`holmes_letters_24`, comme le livre avec *Treasure Island*), envoyées de plusieurs façons. Le code fixe coûte 5 symboles par lettre.
a) `morse_24` : le nombre moyen de points et de traits par lettre avec le Morse (`MORSE`, fourni) (4 décimales) ;
b) `morse_total_24` : le nombre total de symboles en Morse pour tout le texte, en comptant un silence entre deux lettres qui se suivent (✏️ 6.4 d) ;
c) `huffman_24` : le nombre moyen de bits par lettre avec le code de Huffman construit sur les lettres de Holmes : encode tout le texte avec tes fonctions et divise la longueur par le nombre de lettres (4 décimales) ;
d) `gap_24` : l'écart entre c) et l'entropie des lettres de Holmes (4 décimales) ;
e) `ratio_24` : le taux de compression de Huffman par rapport au code fixe (4 décimales) ;
f) `morse_beats_entropy_24` : a) est-il plus petit que l'entropie des lettres ? (`True` ou `False`)

Dans tes notes : f) contredit-il Shannon, pour qui aucun code ne descend en moyenne sous l'entropie ? (Combien de symboles différents le Morse utilise-t-il vraiment ?) La borne « entropie $\le$ longueur moyenne $<$ entropie $+ 1$ » est-elle respectée par c) ?

In [ ]:
MORSE = dict(a=".-", b="-...", c="-.-.", d="-..", e=".", f="..-.", g="--.", h="....", i="..", j=".---",
             k="-.-", l=".-..", m="--", n="-.", o="---", p=".--.", q="--.-", r=".-.", s="...", t="-",
             u="..-", v="...-", w=".--", x="-..-", y="-.--", z="--..")      # international Morse code
holmes_letters_24 = letters_only(holmes)                 # all the letters a-z of Holmes, in order
print(len(holmes_letters_24), "letters; with the 5-symbol fixed code:", 5 * len(holmes_letters_24), "symbols")

In [ ]:
morse_24 = ...                 # a)
morse_total_24 = ...           # b) an integer
huffman_24 = ...               # c)
gap_24 = ...                   # d)
ratio_24 = ...                 # e)
morse_beats_entropy_24 = ...   # f) True or False

In [ ]:
wb.check("6.24a", morse_24)
wb.check("6.24b", morse_total_24)
wb.check("6.24c", huffman_24)
wb.check("6.24d", gap_24)
wb.check("6.24e", ratio_24)
wb.check("6.24f", morse_beats_entropy_24)

### Ex 6.25 — Le code de Huffman de Holmes pour envoyer Verne 🔮 ★★★ ⏱️ 30 min
**Objectif :** prévoir puis mesurer ce que coûte un code de Huffman fait pour une autre langue, et le relier à la KL.  
**Prérequis :** Ex 6.23 · Ex 6.18 · fiche §6.8, §6.9 · **Fil rouge :** Holmes/Verne · **Parcours :** C

On construit deux codes de Huffman sur les 26 lettres : celui des lettres de Holmes et celui des lettres de Verne (a–z, accents ignorés, comme en 6.18). **Avant d'exécuter quoi que ce soit**, prédis :
a) `prediction_6_25a` : envoyer les lettres de Verne avec le code de Holmes coûte-t-il plus de bits par lettre qu'avec le code de Verne ? (`True` ou `False`)
b) `prediction_6_25b` : coûte-t-il plus de 5 bits par lettre, le prix du code fixe ? (`True` ou `False`)
c) `prediction_6_25c` : le surcoût par lettre est le plus proche de quelle valeur : 0,02, 0,2 ou 2 bits ?
d) `prediction_6_25d` : dans l'autre sens (Holmes avec le code de Verne), le surcoût est-il plus grand ? (`True` ou `False`)

Écris ton hypothèse (cellule 📝), puis tes quatre prédictions. Ensuite seulement, exécute l'**Expérience** : elle encode les deux textes avec les deux codes, avec tes fonctions, et vérifie e) le nombre de bits par lettre de Verne avec le code de Holmes et f) le surcoût par rapport au code de Verne.

Dans tes notes : compare f) à $\mathrm{KL}(\text{Verne} \,\|\, \text{Holmes})$ de 6.18 c). Pourquoi ne sont-ils pas exactement égaux ?

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_6_25a = ...   # a) True or False
prediction_6_25b = ...   # b) True or False
prediction_6_25c = ...   # c) 0.02, 0.2 or 2
prediction_6_25d = ...   # d) True or False

In [ ]:
wb.check("6.25a", prediction_6_25a)
wb.check("6.25b", prediction_6_25b)
wb.check("6.25c", prediction_6_25c)
wb.check("6.25d", prediction_6_25d)

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="6")   # reload your saved file
if any(answer is ... or answer is None for answer in [prediction_6_25a, prediction_6_25b, prediction_6_25c, prediction_6_25d]):
    print("✍️ Écris d'abord tes quatre prédictions dans la cellule du dessus.")
else:
    with wb.attempt("6.25"):
        texts_25 = {"Holmes": letters_only(holmes), "Verne": letters_only(verne)}
        codes_25 = {name: mylearn.info.huffman_code(list(LETTERS), mylearn.info.char_distribution(text, alphabet=LETTERS)[1])
                    for name, text in texts_25.items()}
        rates_25 = {(text_name, code_name): len(mylearn.info.huffman_encode(text, codes_25[code_name])) / len(text)
                    for text_name, text in texts_25.items() for code_name in codes_25}
        print("bits per letter        code of Holmes   code of Verne")
        for text_name in texts_25:
            print(f"{text_name:6s} letters      {rates_25[(text_name, 'Holmes')]:12.4f}   {rates_25[(text_name, 'Verne')]:12.4f}")
        print(f"your predictions: a) {prediction_6_25a}   b) {prediction_6_25b}   c) {prediction_6_25c}   d) {prediction_6_25d}")
        wb.check("6.25e", rates_25[("Verne", "Holmes")], computed=True)
        wb.check("6.25f", rates_25[("Verne", "Holmes")] - rates_25[("Verne", "Verne")], computed=True)

### Ex 6.26 — Le contexte local réduit la surprise : les bigrammes 🔬 ★★★ ⏱️ 40 min
**Objectif :** mesurer combien de bits par lettre fait gagner la connaissance de la lettre précédente, sur des données non vues.  
**Prérequis :** Ex 6.22 · fiche §6.2, au-delà du livre (2) (🧮 entropie conditionnelle) · livre §6.2.2 · **Fil rouge :** Holmes/Verne · **Parcours :** M, C

Un modèle **unigramme** donne à chaque lettre la même probabilité, quelle que soit la lettre d'avant. Un modèle **bigramme** prédit une lettre **sachant la précédente** : c'est le contexte local du §6.2.2 (fiche, 🧮 entropie conditionnelle). On apprend les deux sur la première moitié des lettres de Holmes (`train_26`) et on les juge sur la seconde (`test_26`), comme on évalue un modèle sur des données qu'il n'a pas vues (ch. 8).
1. Le modèle unigramme : la distribution `char_distribution(train_26, alphabet=LETTERS, smoothing=1)`.
2. Le modèle bigramme : la matrice `bigram_26`, de forme (26, 26), dont la ligne $a$ est la distribution de la lettre qui suit $a$ dans `train_26` (dans l'ordre de `LETTERS`), lissée par `smoothing=1`. Une façon de faire : pour chaque lettre, la liste des lettres qui la suivent (`zip(train_26, train_26[1:])`), puis `token_distribution(…, vocabulary=list(LETTERS), smoothing=1)`.

Avec tes fonctions (4 décimales, sauf c)) :
a) `uni_bits_26` : la surprise moyenne, en bits, des lettres de `test_26` sous le modèle unigramme, la moyenne de $-\log_2 q(x_t)$ ;
b) `bi_bits_26` : la surprise moyenne sous le modèle bigramme, la moyenne de $-\log_2 P(x_t \mid x_{t-1})$ pour $t = 1, \ldots, n - 1$ (la première lettre n'a pas de précédente : on ne la compte pas) ;
c) `ppl_26` : la liste `[perplexité unigramme, perplexité bigramme]` sur `test_26` (2 décimales) ;
d) `bi_verne_26` : la surprise moyenne du modèle **bigramme de Holmes** sur les lettres de Verne (`verne_letters_26`, même convention qu'en b)).

Puis, si tu veux, un modèle trigramme (★ bonus, non vérifié) : jusqu'où descend-on ?

Dans tes notes : combien de bits par lettre le contexte d'une seule lettre fait-il gagner ? Pourquoi le gain est-il plus petit sur Verne ? Relie ces nombres à zlib (📦 6.19).

In [ ]:
holmes_letters_26 = letters_only(holmes)
half_26 = len(holmes_letters_26) // 2
train_26, test_26 = holmes_letters_26[:half_26], holmes_letters_26[half_26:]   # learn on the first half, judge on the second
verne_letters_26 = letters_only(verne)
print(len(train_26), "letters to learn,", len(test_26), "to judge,", len(verne_letters_26), "letters of Verne")

In [ ]:
bigram_26 = ...     # the (26, 26) matrix of the bigram model: row = previous letter (not checked)
uni_bits_26 = ...   # a)
bi_bits_26 = ...    # b)
ppl_26 = ...        # c) [unigram perplexity, bigram perplexity]
bi_verne_26 = ...   # d)

In [ ]:
wb.check("6.26a", uni_bits_26)
wb.check("6.26b", bi_bits_26)
wb.check("6.26c", ppl_26)
wb.check("6.26d", bi_verne_26)

### Ex 6.27 — Passer sous la barre de Huffman lettre à lettre 🏆 ★★★ ⏱️ 60 min
**Objectif :** dépasser le code de Huffman des lettres en codant des blocs de lettres, sur des données que le code n'a pas vues.  
**Prérequis :** Ex 6.24 · Ex 6.26 · fiche §6.6, au-delà du livre (2) · **Fil rouge :** Holmes · **Parcours :** C

Le livre s'arrête au code de Huffman des **lettres**. Le contexte local (6.26) permet de faire mieux : coder des **blocs** de lettres. Les règles :
- le code est construit sur la première moitié des lettres de Holmes (`TRAIN_27`), et l'on envoie la seconde (`TEST_27`) : l'émetteur et le récepteur partagent la première moitié, comme un contexte global ;
- tu écris `split_27(letters)`, qui découpe une chaîne de lettres en une liste de blocs de 1 à 8 lettres dont la concaténation redonne exactement la chaîne, et `model_27(train_blocks)`, qui renvoie `(symbols, probs)` : la distribution à partir de laquelle `huffman_code` construit le code. `model_27` ne reçoit **que** les blocs de la première moitié ; ni `model_27` ni `split_27` ne lisent `TEST_27` (ou le texte de Holmes) ;
- chaque bloc de la seconde moitié doit avoir un mot de code : ton modèle doit donner une probabilité non nulle à tout bloc qui peut apparaître (lissage de Laplace, 6.17), y compris un bloc plus court à la fin du texte ;
- `grade_27` (fourni) découpe les deux moitiés, construit le code avec **ta** fonction `huffman_code`, encode `TEST_27`, vérifie que le décodage le redonne exactement, et calcule le nombre de bits par lettre.

**Objectif : moins de 3,80 bits par lettre.** Le point de départ fourni, le code de Huffman lettre à lettre, en fait environ 4,2.

In [ ]:
TRAIN_27, TEST_27 = train_26, test_26          # the two halves of the letters of Holmes (6.26)
MAX_BLOCK_27 = 8                                  # a block has 1 to 8 letters


def grade_27(split, model, train=TRAIN_27, test=TEST_27):
    """Build the Huffman code on `train` with your huffman_code, send `test` with it: (bits per letter, message)."""

    def blocks_of(text, half):
        blocks = list(split(text))
        if not all(isinstance(block, str) and 1 <= len(block) <= MAX_BLOCK_27 for block in blocks):
            return None, f"chaque bloc doit être une chaîne de 1 à {MAX_BLOCK_27} lettres (un str, pas un tuple ni une liste)."
        if "".join(blocks) != text:
            return None, (f"split_27 doit découper le texte sans rien perdre ni ajouter (sur la {half} moitié) : "
                          "\"\".join(blocs) == texte.")
        return blocks, "ok"

    blocks_train, message = blocks_of(train, "première")
    if blocks_train is None:
        return None, message
    symbols, probs = model(blocks_train)                       # the code is built BEFORE the test half is split
    code = mylearn.info.huffman_code(list(symbols), probs)
    blocks_test, message = blocks_of(test, "seconde")
    if blocks_test is None:
        return None, message
    missing = sorted({block for block in blocks_test if block not in code})
    if missing:
        return None, (f"{len(missing)} bloc(s) de la seconde moitié sans mot de code, dont {missing[:3]} : ton "
                      "modèle doit donner une probabilité non nulle à tout bloc possible (lissage).")
    bits = "".join(code[block] for block in blocks_test)       # encoded and decoded here, independently of you
    inverse = {word: symbol for symbol, word in code.items()}
    decoded, current = [], ""
    for bit in bits:
        current += bit
        if current in inverse:
            decoded.append(inverse[current])
            current = ""
    if current or "".join(decoded) != test:
        return None, "le décodage ne redonne pas exactement la seconde moitié : le code est-il bien préfixe ?"
    return len(bits) / len(test), "ok"

In [ ]:
def split_27(letters):
    """Cut a string of letters into blocks whose concatenation is exactly `letters`."""
    return list(letters)                                  # the starting point: one block per letter


def model_27(train_blocks):
    """(symbols, probs): every block that can appear, with its probability, learned from the training blocks only."""
    return mylearn.info.token_distribution(train_blocks, vocabulary=list(LETTERS), smoothing=1)

In [ ]:
with wb.attempt("6.27"):
    rate_27, message_27 = grade_27(split_27, model_27)
    if rate_27 is None:
        print("❌ Ex 6.27 :", message_27)
    elif rate_27 < 3.80:
        print(f"✅ Ex 6.27 : défi réussi, {rate_27:.4f} bits par lettre (objectif : moins de 3,80).")
    else:
        print(f"❌ Ex 6.27 : {rate_27:.4f} bits par lettre : il faut descendre sous 3,80.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une surprise, une entropie, une cross-entropy et une KL, en bits et en nats ?
2. Sais-tu construire un code de Huffman, et expliquer pourquoi il est préfixe et proche de l'entropie ?
3. Sais-tu relier la log loss d'un classifieur et la perplexité d'un modèle de langage à la cross-entropy ?

**Pour aller plus loin** : les références de la fiche (l'article de Shannon, « Visual Information Theory » de C. Olah, la vidéo de 3Blue1Brown sur Wordle). La suite : le checkpoint de la partie I et son mini-projet, un détecteur de langue anglais / français ; puis le ch. 13, où l'entropie choisit les questions d'un arbre de décision, et le ch. 18, où la cross-entropy devient la loss que l'on rétropropage.